# 第3回 AR・MA・ARMA

このノートブックは、スライドと同じ順に並んでいます。
コードはすべて完成しているので、上から順に実行して、結果を読んでください。

- 0節：セットアップ（授業の最初に実行する）
- A節：理論をシミュレーションで確かめる
- B節：実データを解析する
- C節：設定を変えて回し直す

今日の問い：昨日までの値から今日の値が決まる式を、データからどう選ぶか。

## 0. セットアップ

授業が始まったら、まずこのセルを実行してください。
パッケージのインストールに少し時間がかかります。終わるまで、次のセルは動きません。

In [ ]:
# OSのコードネームを確認（noble = Ubuntu 24.04 など）
system("lsb_release -cs", intern = TRUE)

options(
  repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/noble/2026-09-01"),
  HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
    paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"]))
)
install.packages(c("tsibble", "fable", "feasts"))
sessionInfo()

## A. 理論をシミュレーションで確かめる

In [ ]:
library(tsibble)
library(fable)
library(feasts)
library(ggtime)   # ACFと予測の図（autoplot）
library(dplyr)
library(ggplot2)
options(repr.plot.width = 8, repr.plot.height = 4.5)

### A-1. 2つの作り方：ARとMA

今日のモデルは、ホワイトノイズ $w_t$（分散 $\sigma_w^2$）から系列 $x_t$ を作る式です。作り方は2つあります。

- 自己回帰（autoregressive, AR）モデル：過去の値を使う。AR(p)は $x_t = \phi_1 x_{t-1} + \dots + \phi_p x_{t-p} + w_t$
- 移動平均（moving average, MA）モデル：過去のノイズを使う。MA(q)は $x_t = w_t + \theta_1 w_{t-1} + \dots + \theta_q w_{t-q}$
- 両方を使うものを、自己回帰移動平均（autoregressive moving average, ARMA）モデルと呼びます。ARMA(p, q)と書きます。

今日は、次の4つを例にします。

In [ ]:
models <- list(
  "1 AR(1)"     = list(ar = 0.7,          ma = numeric(0)),   # x_t = 0.7 x_{t-1} + w_t
  "2 MA(1)"     = list(ar = numeric(0),   ma = 0.7),          # x_t = w_t + 0.7 w_{t-1}
  "3 AR(2)"     = list(ar = c(1.0, -0.5), ma = numeric(0)),   # x_t = x_{t-1} - 0.5 x_{t-2} + w_t
  "4 ARMA(1,1)" = list(ar = 0.7,          ma = 0.5)           # x_t = 0.7 x_{t-1} + w_t + 0.5 w_{t-1}
)

ある時点で、ノイズが1回だけ1になったとします（これをショックと呼びます）。そのショックが、$j$ 時点あとの値にどれだけ残るかを $\psi_j$ と書きます。

In [ ]:
shock <- do.call(rbind, lapply(names(models), function(m)
  data.frame(model = m, j = 0:15,
             psi = c(1, ARMAtoMA(models[[m]]$ar, models[[m]]$ma, lag.max = 15)))))
ggplot(shock, aes(j, psi)) +
  geom_hline(yintercept = 0, colour = "grey60") +
  geom_col(width = 0.15) +
  geom_point() +
  facet_wrap(~model) +
  labs(x = "Steps after the shock j", y = "Effect of one shock (psi_j)")

- AR(1)：ショックは、1時点ごとに0.7倍になって、いつまでも残ります。
- MA(1)：ショックは、次の時点まで残り、そのあとは消えます。
- AR(2)：ショックは、波を打ちながら小さくなります。
- ARMA(1,1)：最初だけMAの分が上乗せされ、そのあとはAR(1)と同じ割合で小さくなります。

4つのモデルから、長さ300の系列を1本ずつ作ります。

In [ ]:
set.seed(7)
n <- 300
sim <- do.call(rbind, lapply(names(models), function(m)
  data.frame(model = m, t = 1:n, x = as.numeric(arima.sim(models[[m]], n = n)))))
ggplot(sim, aes(t, x)) +
  geom_line() +
  facet_wrap(~model, ncol = 1, scales = "free_y") +
  labs(x = "Time t (step)", y = "Value x_t")

系列の図だけでは、4つを見分けにくいことを確かめてください。

### A-2. ACFとPACFは、モデルの指紋

モデルを決めると、自己相関関数（ACF）$\rho(h)$ が式で決まります。これを理論ACFと呼びます。
標本ACF（棒）に、理論ACF（赤い点）を重ねます。青い破線は、第1回と同じ $\pm 1.96/\sqrt{n}$ です。

In [ ]:
# 標本ACFと理論ACFを、ラグごとに並べた表にする（type = "pacf" なら偏自己相関）
acf_compare <- function(x, ar = numeric(0), ma = numeric(0), lag_max = 20, type = "acf") {
  white <- length(ar) + length(ma) == 0     # 係数が1つもなければ、ホワイトノイズ（理論値はすべて0）
  if (type == "acf") {
    sample_value <- acf(x, lag.max = lag_max, plot = FALSE)$acf[-1]
    theory_value <- if (white) rep(0, lag_max) else ARMAacf(ar, ma, lag.max = lag_max)[-1]
  } else {
    sample_value <- pacf(x, lag.max = lag_max, plot = FALSE)$acf[, 1, 1]
    theory_value <- if (white) rep(0, lag_max) else ARMAacf(ar, ma, lag.max = lag_max, pacf = TRUE)
  }
  data.frame(lag = 1:lag_max, sample = as.numeric(sample_value),
             theory = as.numeric(theory_value), n = length(x))
}
# 図にする（棒：標本、赤い点：理論、青い破線：±1.96/√n）
plot_compare <- function(d, y_lab = "ACF (bar: sample, red: theory)") {
  ggplot(d, aes(lag)) +
    geom_hline(yintercept = 0, colour = "grey60") +
    geom_col(aes(y = sample), width = 0.15) +
    geom_point(aes(y = theory), colour = "red", size = 1.8) +
    geom_hline(yintercept = c(-1, 1) * 1.96 / sqrt(d$n[1]), colour = "blue", linetype = "dashed") +
    labs(x = "Lag h (step)", y = y_lab)
}

In [ ]:
cmp_acf <- do.call(rbind, lapply(names(models), function(m)
  cbind(model = m, acf_compare(sim$x[sim$model == m], models[[m]]$ar, models[[m]]$ma))))
plot_compare(cmp_acf) +
  facet_wrap(~model)

- AR(1)：$\rho(h) = 0.7^h$。ラグとともに、少しずつ小さくなります。
- MA(1)：ラグ1だけが0でなく、ラグ2から先は0です。ここで切れます。
- AR(2)：波を打ちながら小さくなります。
- ARMA(1,1)：少しずつ小さくなります。

棒は赤い点の近くにあります。ずれは、データが300個しかないことによる誤差です。

偏自己相関関数（PACF）でも同じことをします。

In [ ]:
cmp_pacf <- do.call(rbind, lapply(names(models), function(m)
  cbind(model = m, acf_compare(sim$x[sim$model == m], models[[m]]$ar, models[[m]]$ma, type = "pacf"))))
plot_compare(cmp_pacf, "PACF (bar: sample, red: theory)") +
  facet_wrap(~model)

- AR(1)：ラグ1で切れます。AR(2)：ラグ2で切れます。
- MA(1)とARMA(1,1)：切れずに、少しずつ小さくなります。

| モデル | ACF | PACF |
| --- | --- | --- |
| AR(p) | 少しずつ小さくなる | ラグpで切れる |
| MA(q) | ラグqで切れる | 少しずつ小さくなる |
| ARMA(p, q) | 少しずつ小さくなる | 少しずつ小さくなる |

この表を使って、モデルを当ててみます。次のセルは、3本の系列A、B、Cを作ります。
スライドの図を見て、それぞれがAR(1)、AR(2)、MA(2)のどれかを予想してから、実行してください。赤い点が答えです。

In [ ]:
set.seed(3)
mystery <- list(
  "A" = list(ar = numeric(0),  ma = c(0.8, 0.5)),   # MA(2)
  "B" = list(ar = -0.7,        ma = numeric(0)),    # AR(1)、係数が負
  "C" = list(ar = c(0.4, 0.4), ma = numeric(0))     # AR(2)
)
quiz <- do.call(rbind, lapply(names(mystery), function(m) {
  x <- as.numeric(arima.sim(mystery[[m]], n = 500))
  rbind(cbind(series = m, type = "ACF",  acf_compare(x, mystery[[m]]$ar, mystery[[m]]$ma, 12)),
        cbind(series = m, type = "PACF", acf_compare(x, mystery[[m]]$ar, mystery[[m]]$ma, 12, type = "pacf")))
}))
plot_compare(quiz, "ACF / PACF (bar: sample, red: theory)") +
  scale_x_continuous(breaks = seq(2, 12, 2)) +
  facet_grid(type ~ series)

### A-3. 定常条件

AR(1)の係数 $\phi$ を、0.5、0.9、1、1.02と変えます。ノイズは4本とも同じです。

In [ ]:
set.seed(1)
w <- rnorm(200)
paths <- do.call(rbind, lapply(c(0.5, 0.9, 1, 1.02), function(phi)
  data.frame(phi = sprintf("phi = %.2f", phi), t = 1:200,
             x = as.numeric(stats::filter(w, phi, method = "recursive")))))
ggplot(paths, aes(t, x)) +
  geom_line() +
  facet_wrap(~phi, scales = "free_y") +
  labs(x = "Time t (step)", y = "Value x_t")

- $|\phi| < 1$：同じ高さのまわりを動きます（定常）。
- $\phi = 1$：ランダムウォークです。高さが定まりません。
- $\phi > 1$：ショックが大きくなりながら残り、系列は発散します。

AR(p)が定常になる条件は、次の方程式の根 $z$ の絶対値が、すべて1より大きいことです。

$$
1 - \phi_1 z - \phi_2 z^2 - \dots - \phi_p z^p = 0
$$

AR(1)なら根は $z = 1/\phi$ なので、$|\phi| < 1$ と同じ条件です。根の絶対値を出す関数を作ります。

In [ ]:
# AR多項式 1 - phi_1 z - ... - phi_p z^p の根の絶対値（すべて1より大きければ、定常）
ar_roots <- function(ar) Mod(polyroot(c(1, -ar)))
# MA多項式 1 + theta_1 z + ... + theta_q z^q の根の絶対値（すべて1より大きければ、反転可能）
ma_roots <- function(ma) Mod(polyroot(c(1, ma)))

AR(2)の係数を4組試します。同じノイズから系列を作り、絶対値の最大も見ます。

In [ ]:
set.seed(1)
w <- rnorm(200)
cand2 <- list(c(1.0, -0.5), c(0.4, 0.4), c(0.6, 0.5), c(1.0, -1.05))
data.frame(
  phi1 = sapply(cand2, `[`, 1),
  phi2 = sapply(cand2, `[`, 2),
  min_root = round(sapply(cand2, function(a) min(ar_roots(a))), 3),          # 根の絶対値の小さいほう
  stationary = sapply(cand2, function(a) all(ar_roots(a) > 1)),
  max_abs_x = signif(sapply(cand2, function(a) max(abs(stats::filter(w, a, method = "recursive")))), 3)
)

根の絶対値が1より小さい組では、長さ200のうちに値が大きくふくらみます。

### A-4. 反転可能性

MA(1)で、$\theta = 0.5$（ノイズの標準偏差1）と $\theta = 2$（ノイズの標準偏差0.5）を比べます。

In [ ]:
twins <- rbind(
  c(1^2   * (1 + 0.5^2), ARMAacf(ma = 0.5, lag.max = 3)[-1]),   # theta = 0.5、ノイズの標準偏差1
  c(0.5^2 * (1 + 2^2),   ARMAacf(ma = 2,   lag.max = 3)[-1]))   # theta = 2、ノイズの標準偏差0.5
dimnames(twins) <- list(c("theta = 0.5", "theta = 2"), c("variance", "rho(1)", "rho(2)", "rho(3)"))
twins

分散もACFも、まったく同じです。データの平均・分散・自己相関からは、2つを区別できません。

どちらを選ぶかを決めるために、観測した値からノイズを復元してみます。
MA(1)の式を書き替えると $w_t = x_t - \theta\,w_{t-1}$ なので、$w_0 = 0$ と仮に決めれば、順に計算できます。

In [ ]:
# MA(1)のノイズを、観測した値から順に復元する（w_0 = 0 から始める）
recover_noise <- function(x, theta) {
  w_hat <- numeric(length(x))
  prev <- 0
  for (t in seq_along(x)) {
    w_hat[t] <- x[t] - theta * prev
    prev <- w_hat[t]
  }
  w_hat
}

In [ ]:
set.seed(1)
m <- 40
w1 <- rnorm(m + 1)                # theta = 0.5 のノイズ（標準偏差1）
x1 <- w1[-1] + 0.5 * w1[-(m + 1)]
w2 <- rnorm(m + 1, sd = 0.5)      # theta = 2 のノイズ（標準偏差0.5）
x2 <- w2[-1] + 2 * w2[-(m + 1)]
err <- rbind(
  data.frame(theta = "theta = 0.5", t = 1:m, error = abs(recover_noise(x1, 0.5) - w1[-1])),
  data.frame(theta = "theta = 2",   t = 1:m, error = abs(recover_noise(x2, 2)   - w2[-1])))
ggplot(err, aes(t, error)) +
  geom_line() +
  scale_y_log10() +
  facet_wrap(~theta) +
  labs(x = "Time t (step)", y = "|recovered noise - true noise| (log scale)")

- $\theta = 0.5$：最初に $w_0$ を仮に0としたことによる誤差は、1時点ごとに半分になり、消えていきます。
- $\theta = 2$：誤差は1時点ごとに2倍になります。ノイズを復元できません。

過去の値からノイズを復元できることを、反転可能（invertible）といいます。MA(1)では $|\theta| < 1$ が条件です。
同じACFをもつ2つのうち、反転可能なほうを選ぶ、と決めておきます。

### A-5. 係数を推定する

ARモデルは、「1時点前の値」を説明変数にした回帰の形をしています。最小二乗法（least squares）で係数を推定できます。

In [ ]:
ar1 <- sim$x[sim$model == "1 AR(1)"]
lagged <- data.frame(x = ar1[-1], x_prev = ar1[-n])    # x_t と x_{t-1} を横に並べる
coef(lm(x ~ 0 + x_prev, data = lagged))
ggplot(lagged, aes(x_prev, x)) +
  geom_point(alpha = 0.4) +
  geom_smooth(method = "lm", formula = y ~ 0 + x, se = FALSE, colour = "red") +
  labs(x = "x_{t-1}", y = "x_t")

傾きが $\phi$ の推定値です。本当の値は0.7です。

MAモデルでは、説明変数にあたる過去のノイズが見えません。回帰はそのままでは使えません。
そこで、$\theta$ を仮に決めてノイズを復元し（A-4の関数）、その2乗和が最も小さくなる $\theta$ を探します。

In [ ]:
ma1 <- sim$x[sim$model == "2 MA(1)"]
css <- data.frame(theta = seq(-0.95, 0.95, 0.01))
css$ss <- sapply(css$theta, function(th) sum(recover_noise(ma1, th)^2))
css$theta[which.min(css$ss)]
ggplot(css, aes(theta, ss)) +
  geom_line() +
  geom_vline(xintercept = 0.7, colour = "red", linetype = "dashed") +
  labs(x = "theta", y = "Sum of squared recovered noise")

赤い破線が、本当の値0.7です。2乗和は、その近くで最も小さくなります。

実際の解析では、最尤法（maximum likelihood）を使います。ノイズが正規分布に従うとして、手元のデータが最も出やすくなる係数を選ぶ方法です。
`fable`パッケージの`ARIMA()`は、最尤法で係数を推定します。`pdq(p, 0, q)`の`p`がARの次数、`q`がMAの次数です（まん中の0は第4回で使います）。
`x ~ 0`の`0`は、平均を0とする、という意味です。

In [ ]:
sim_ts <- as_tsibble(sim, key = model, index = t)
fits <- list(
  sim_ts |> filter(model == "1 AR(1)")     |> model(arma = ARIMA(x ~ 0 + pdq(1, 0, 0))),
  sim_ts |> filter(model == "2 MA(1)")     |> model(arma = ARIMA(x ~ 0 + pdq(0, 0, 1))),
  sim_ts |> filter(model == "3 AR(2)")     |> model(arma = ARIMA(x ~ 0 + pdq(2, 0, 0))),
  sim_ts |> filter(model == "4 ARMA(1,1)") |> model(arma = ARIMA(x ~ 0 + pdq(1, 0, 1)))
)
est <- do.call(rbind, lapply(fits, function(f) as.data.frame(tidy(f))[, c("model", "term", "estimate", "std.error")]))
est$true <- c(0.7, 0.7, 1.0, -0.5, 0.7, 0.5)
est[, 3:4] <- round(est[, 3:4], 3)
est

推定値（`estimate`）は、本当の値（`true`）から、標準誤差（`std.error`）の2倍ほどの範囲に入っています。

### A-6. 次数を選ぶ

ここまでは、正しい次数を知っているとして推定しました。実データでは、次数も分かりません。

AR(2)から作った系列に、AR(0)からAR(5)までを当てはめます。

In [ ]:
ar2_ts <- sim_ts |> filter(model == "3 AR(2)")
fits_p <- ar2_ts |>
  model(p0 = ARIMA(x ~ 0 + pdq(0, 0, 0)),
        p1 = ARIMA(x ~ 0 + pdq(1, 0, 0)),
        p2 = ARIMA(x ~ 0 + pdq(2, 0, 0)),
        p3 = ARIMA(x ~ 0 + pdq(3, 0, 0)),
        p4 = ARIMA(x ~ 0 + pdq(4, 0, 0)),
        p5 = ARIMA(x ~ 0 + pdq(5, 0, 0)))
ic <- as.data.frame(glance(fits_p))[, c(".model", "sigma2", "log_lik", "AIC", "BIC")]
ic$p <- 0:5
ic$minus2loglik <- -2 * ic$log_lik
round(ic[, c("p", "sigma2", "minus2loglik", "AIC", "BIC")], 2)

- `minus2loglik`（$-2 \times$ 対数尤度）は、当てはまりの悪さです。次数を上げるほど、必ず小さくなります。これだけでは、次数を選べません。
- 赤池情報量規準（Akaike information criterion, AIC）は、パラメータの数 $k$ の2倍を足します。$\mathrm{AIC} = -2 \log L + 2k$
- ベイズ情報量規準（Bayesian information criterion, BIC）は、$k \log n$ を足します。$\mathrm{BIC} = -2 \log L + k \log n$

どちらも、小さいほど良いモデルです。

In [ ]:
ic_long <- rbind(data.frame(p = ic$p, measure = "-2 log-likelihood", value = ic$minus2loglik),
                 data.frame(p = ic$p, measure = "AIC", value = ic$AIC),
                 data.frame(p = ic$p, measure = "BIC", value = ic$BIC))
ggplot(subset(ic_long, p >= 1), aes(p, value, colour = measure)) +
  geom_line() +
  geom_point() +
  labs(x = "AR order p", y = "Value (smaller is better)", colour = NULL)
c(AIC = ic$p[which.min(ic$AIC)], BIC = ic$p[which.min(ic$BIC)])

図では、$p = 0$ を省いています（値が大きく、ほかの違いが見えなくなるため）。

1本だけでは、たまたまかもしれません。長さ100のAR(2)を200本作り、AICとBICが選ぶ次数を数えます。

In [ ]:
set.seed(1)
pick <- replicate(200, {
  x <- arima.sim(list(ar = c(1.0, -0.5)), n = 100)
  f <- lapply(0:5, function(p) arima(x, order = c(p, 0, 0), include.mean = FALSE, method = "ML"))
  c(AIC = which.min(sapply(f, AIC)) - 1, BIC = which.min(sapply(f, BIC)) - 1)
})
picked <- rbind(AIC = table(factor(pick["AIC", ], levels = 0:5)),
                BIC = table(factor(pick["BIC", ], levels = 0:5)))
picked

列が、選ばれた次数です。本当の次数は2です。

- どちらも、本当の次数より小さい次数は、ほとんど選びません。
- AICは、大きすぎる次数を選ぶことがあります。BICは罰則が重いので、それが少なくなります。

### A-7. 予測する

ARMAモデルの予測を、第2回のベンチマーク（平均法、ナイーブ法）と比べます。
平均が10のAR(1)（$\phi = 0.8$）を作り、最後の20個をテストデータにします。`y ~ 1`の`1`は、平均を推定する、という意味です。

In [ ]:
set.seed(1)
n_fc <- 200
h <- 20
sim_fc <- tsibble(t = 1:n_fc, y = 10 + as.numeric(arima.sim(list(ar = 0.8), n = n_fc)), index = t)
train <- sim_fc |> filter(t <= n_fc - h)
fit <- train |>
  model(mean  = MEAN(y),
        naive = NAIVE(y),
        ar1   = ARIMA(y ~ 1 + pdq(1, 0, 0)))
fc <- fit |> forecast(h = h)
autoplot(fc, sim_fc |> filter(t > n_fc - 4 * h), level = NULL) +
  labs(x = "Time t (step)", y = "Value y_t", colour = "Method")
accuracy(fc, sim_fc) |> select(.model, RMSE, MAE)

AR(1)の予測は、最後の値から始まり、先へ行くほど平均に近づきます。ナイーブ法（最後の値のまま）と平均法（平均のまま）のあいだを通ります。

1本だけでは、たまたまかもしれません。長さ100の系列を1000本作り、1〜10時点先の予測の誤差（RMSE）を、時点ごとに出します。
線は、係数と平均が分かっているときの理論値です。

In [ ]:
set.seed(1)
phi <- 0.8
H <- 10
err <- replicate(1000, {
  x <- as.numeric(arima.sim(list(ar = phi), n = 100 + H))
  x_train <- x[1:100]
  x_test <- x[101:(100 + H)]
  f <- arima(x_train, order = c(1, 0, 0), method = "ML")
  rbind(mean  = mean(x_train) - x_test,
        naive = x_train[100] - x_test,
        ar1   = as.numeric(predict(f, n.ahead = H)$pred) - x_test)
})
rmse_h <- sqrt(apply(err^2, c(1, 2), mean))      # 行：方法、列：何時点先か
gamma0 <- 1 / (1 - phi^2)                        # 系列の分散（ノイズの分散は1）
by_h <- rbind(
  data.frame(method = "mean",  h = 1:H, simulated = rmse_h["mean", ],  theory = sqrt(gamma0)),
  data.frame(method = "naive", h = 1:H, simulated = rmse_h["naive", ], theory = sqrt(2 * gamma0 * (1 - phi^(1:H)))),
  data.frame(method = "ar1",   h = 1:H, simulated = rmse_h["ar1", ],   theory = sqrt(gamma0 * (1 - phi^(2 * (1:H))))))
ggplot(by_h, aes(h, colour = method)) +
  geom_line(aes(y = theory)) +
  geom_point(aes(y = simulated)) +
  scale_x_continuous(breaks = 1:H) +
  labs(x = "Forecast horizon h (step)", y = "RMSE (point: simulated, line: theory)", colour = "Method")
round(rmse_h[, c(1, 2, 5, 10)], 2)

- 1時点先：AR(1)とナイーブ法は近く、平均法が大きく外れます。
- 10時点先：AR(1)と平均法は近く、ナイーブ法が大きく外れます。
- AR(1)は、どの時点でも最も小さくなります。データを作ったモデルと同じ形の式で予測しているからです。

## B. 実データを解析する

Rに付属する`LakeHuron`を使います。北アメリカのヒューロン湖の、年ごとの水位です。

B節は、A節で作った関数（`acf_compare`、`plot_compare`、`ar_roots`、`ma_roots`）を使います。A節を実行してから進んでください。

### B-0. 設定

このセルの値を変えると、B節の解析がすべて変わります。まずはこのまま実行してください。

In [ ]:
# ここを変えると、以下の解析がすべて変わる
year_from <- 1875     # 始まりの年：1875〜1890
year_to   <- 1972     # 終わりの年：1960〜1972
p         <- 2        # 自分で選ぶモデルの、ARの次数：0〜3
q         <- 0        # 自分で選ぶモデルの、MAの次数：0〜2（p + q は3以下にする）
h         <- 10       # テストデータの年数：5 か 10

### B-1. データを確かめる

`LakeHuron`は、Rの古い形式（`ts`）の時系列です。`tsibble`に直してから確かめます。

In [ ]:
class(LakeHuron)
c(start = start(LakeHuron)[1], end = end(LakeHuron)[1], frequency = frequency(LakeHuron))
lake_all <- as_tsibble(LakeHuron) |>
  rename(year = index, level = value)
glimpse(lake_all)
interval(lake_all)
c(rows = nrow(lake_all), missing = sum(is.na(lake_all$level)), gaps = has_gaps(lake_all)$.gaps)

- 1行は1年です。`level`が水位（フィート）です。
- 期間は1875年から1972年、98行です。
- 欠測（`NA`）も、行の抜けもありません。

### B-2. 前処理

期間を絞り、最後の`h`年をテストデータにします。年ごとの値なので、季節はありません。

In [ ]:
lake <- lake_all |>
  filter(year >= year_from, year <= year_to) |>
  mutate(y = level)
n <- nrow(lake)
train <- lake |> filter(year <= year_to - h)
c(n = n, train = nrow(train), test = n - nrow(train))

In [ ]:
ggplot(lake, aes(year, y)) +
  annotate("rect", xmin = year_to - h + 0.5, xmax = year_to + 0.5, ymin = -Inf, ymax = Inf,
           alpha = 0.15, fill = "red") +
  geom_line() +
  geom_hline(yintercept = mean(train$y), colour = "grey50", linetype = "dashed") +
  labs(x = "Year", y = "Level of Lake Huron (feet)")

赤い帯がテストデータ、灰色の破線が訓練データの平均です。

### B-3. ACFとPACFから、次数の見当をつける

In [ ]:
train |>
  ACF(y, lag_max = 20) |>
  autoplot() +
  labs(x = "Lag h (year)", y = "Sample ACF")

In [ ]:
train |>
  PACF(y, lag_max = 20) |>
  autoplot() +
  labs(x = "Lag h (year)", y = "Sample PACF")

A-2の表と見比べます。ACFは少しずつ小さくなり、PACFは短いラグで切れているように見えます。ARモデルが候補になります。

### B-4. 自分で選んだモデルを当てはめる

設定セルの`p`と`q`で、ARMA(p, q)を当てはめます。

In [ ]:
fit_my <- train |>
  model(my = ARIMA(y ~ 1 + pdq(p, 0, q)))
coef_my <- as.data.frame(tidy(fit_my))[, c("term", "estimate", "std.error")]
coef_my
ar_hat <- coef_my$estimate[grepl("^ar", coef_my$term)]    # ARの係数
ma_hat <- coef_my$estimate[grepl("^ma", coef_my$term)]    # MAの係数

`constant`は定数項 $c$ です。系列の平均 $\mu$ とは、$c = \mu\,(1 - \phi_1 - \dots - \phi_p)$ の関係があります。

推定したモデルが定常か、反転可能かを、根の絶対値で確かめます（すべて1より大きければよい）。

In [ ]:
list(ar_roots = ar_roots(ar_hat), ma_roots = ma_roots(ma_hat))

推定したモデルの理論ACF（赤い点）を、データの標本ACF（棒）に重ねます。

In [ ]:
plot_compare(acf_compare(train$y, ar_hat, ma_hat, lag_max = 20)) +
  labs(x = "Lag h (year)", y = "ACF (bar: data, red: fitted model)")

赤い点が棒から離れているラグでは、モデルがデータの従属性を写せていません。

残差（1年先の予測の誤差）のACFも見ます。モデルが従属性を写せていれば、残差はホワイトノイズに近くなります。

In [ ]:
fit_my |>
  augment() |>
  ACF(.innov, lag_max = 20) |>
  autoplot() +
  labs(x = "Lag h (year)", y = "Sample ACF of residuals")

### B-5. 候補を並べて、規準で選ぶ

$p = 0, \dots, 3$ と $q = 0, 1, 2$ のうち、$p + q \le 3$ の9通りを当てはめ、AICとBICを並べます。
次数を上げすぎると、98個のデータでは係数をうまく推定できなくなるので、合計を3までにしています。

In [ ]:
orders <- subset(expand.grid(p = 0:3, q = 0:2), p + q <= 3)
cand <- do.call(rbind, lapply(seq_len(nrow(orders)), function(i) {
  p_i <- orders$p[i]
  q_i <- orders$q[i]
  g <- train |> model(ARIMA(y ~ 1 + pdq(p_i, 0, q_i))) |> glance()
  data.frame(p = p_i, q = q_i, sigma2 = g$sigma2, log_lik = g$log_lik, AIC = g$AIC, BIC = g$BIC)
}))
cand <- cand[order(cand$AIC), ]
rownames(cand) <- NULL
round(head(cand, 6), 2)
p_best <- cand$p[1]
q_best <- cand$q[1]
c(p_best = p_best, q_best = q_best)

表は、AICの小さい順です。1行目が、AICで選ばれたモデルです。BICの列も見て、BICなら何行目が選ばれるかを確かめてください。

### B-6. 予測を比べる

訓練データで当てはめ、テストデータの`h`年を予測します。比べるのは次の4つです。

- `mean`：平均法、`naive`：ナイーブ法（第2回のベンチマーク。年ごとの値で季節がないので、季節ナイーブ法は使いません）
- `my`：自分で選んだARMA(p, q)
- `auto`：AICで選ばれたARMA(p_best, q_best)

In [ ]:
fit <- train |>
  model(mean  = MEAN(y),
        naive = NAIVE(y),
        my    = ARIMA(y ~ 1 + pdq(p, 0, q)),
        auto  = ARIMA(y ~ 1 + pdq(p_best, 0, q_best)))
fc <- fit |> forecast(h = h)
autoplot(fc, lake |> filter(year > year_to - 4 * h), level = NULL) +
  labs(x = "Year", y = "Level of Lake Huron (feet)", colour = "Method")

In [ ]:
acc <- accuracy(fc, lake) |> select(.model, RMSE, MAE)
acc

### B-7. 時系列交差検証

訓練データを`h`年ずつ伸ばしながら、同じ評価を4回くり返します。`my`と`auto`の次数は、B-6と同じものを使います。

In [ ]:
acc_cv <- lake |>
  filter(year <= year_to - h) |>
  stretch_tsibble(.init = n - 4 * h, .step = h) |>
  model(mean  = MEAN(y),
        naive = NAIVE(y),
        my    = ARIMA(y ~ 1 + pdq(p, 0, q)),
        auto  = ARIMA(y ~ 1 + pdq(p_best, 0, q_best))) |>
  forecast(h = h) |>
  accuracy(lake) |>
  select(.model, RMSE, MAE)
acc_cv

読み方：

- B-4：短いラグでは、モデルの理論ACFはデータの標本ACFと合います。長いラグで合わなければ、水位の平均がゆっくり動いている（定常でない）のかもしれません。これは第4回で扱います。
- B-5：AICやBICの差が2より小さいモデルは、ほとんど同じ良さです。1位だけを見ないでください。
- B-6、B-7：AICで選んだモデルが、予測でも最も良いとは限りません。それでも、ARMAモデルが2つのベンチマークより良いかどうかを確かめてください。

## C. 設定を変えて回し直す

B-0の設定セルを書き替えて、B節を上からすべて実行し直してください
（Colabでは、設定セルを選んで、メニューの［ランタイム］→［以降のセルを実行］）。

変えられる値：

| 変数 | 値 |
| --- | --- |
| `year_from` | 1875〜1890 |
| `year_to` | 1960〜1972 |
| `p` | 0〜3 |
| `q` | 0〜2（`p + q`は3以下） |
| `h` | 5、10 |

読むときの観点：

1. `p`と`q`を変えると、B-4の理論ACF（赤い点）はどう変わるか。棒に最もよく重なる次数はどれか。
2. `p = 0`、`q = 1`（MA(1)）にすると、残差のACFに何が残るか。予測はどうなるか。
3. 期間（`year_from`、`year_to`）を変えると、推定した係数と、選ばれるモデルは変わるか。
4. `year_to`を1960にすると、B-6でARMAモデルはベンチマークに勝つか。B-7ではどうか。
5. `h`を5にすると、ARMAモデルとベンチマークの差は、広がるか、縮まるか。A-7の最後の図と合わせて読む。

正解は1つではありません。自分の設定で出た結果を、最初の設定（1875〜1972年、AR(2)、10年先）の結果と比べてください。